# BrainDecoding — Broderick 2018 英语 EEG 留出测试集 (`test`) 评估、因果去偏审计与四模型配对检验工作流

本 Notebook 专用于 **Google Colab GPU 实例** 完成 **Broderick et al. (2018)**（`broderick2018`，`all-n19`，`SEED = '0,1,2'`）的**留出测试集（`split=test`）词级检索评估与因果去偏审计**：

1. **高速解包缓存（智能 0 秒跳过）**：当全部目标模型尚未具备最新 `evaluation_test.json` 时，从 Google Drive `archives/features_broderick2018.tar` 秒级解包至 `/content/local_cache`，并通过 `/content/studies_links` 软链接桥接；若已完整缓存报告且未开启 `FORCE_REEVAL`，则自动 0 秒跳过 `.tar` 解包。
2. **默认 4 模型完整闭环（`EVAL_MODE = 'full_bspc_suite'`）**：一键对齐评估 BSPC 主表的全部 4 类代表性模型（`word_only`、`context_unfrozen`、`context_debiased`、`prefix_revision`），亦可切换 `EVAL_MODE = 'single_config'` 单独审计任意一个配置（含 `broderick2018_amortized_timing_prefix_revision.yaml` 档位 B）。
3. **无环绕 Pre-Roll 负对照与在线流式 Causal EMA（`M_eval=0`, `1x` 前向）**：默认使用 `configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml`（Pre-Roll `[-10, -20, -30, -40]s` + `causal_cms:eta=0.5`），可选开启 `RUN_ZERO_SIGNAL_AUDIT = True` 单独跑全零神经信号负对照（隔离写入 `zero_signal_audit/`）。
4. **3 秒 `.pt` 缓存复用 `M_eval` 权衡审计与四模型配对统计检验导出**：自动读取 `subspace_encoded_cache_test.pt` 按实际移位池规模自适应生成 `M_eval` 权衡曲线（`common_mode_subtraction_evaluation`），并自动调用 `upgrade_and_refresh_saved_multi_seed_reports` + `resolve_and_export_cross_model_comparison` 永久导出 `multi_seed_cross_model_comparison_test.{md,txt,json}`。
5. **契约裁剪**：按 BSPC 审稿防御设计，Broderick 2018 聚焦词级检索与因果去偏机制跨模态验证，**无需加载 `Qwen3-8B` 运行 Stage 3 序列束搜索**。

In [ ]:
# 1. 检查 GPU、挂载 Google Drive、配置 Broderick 2018 评估队列与环境变量
import os
import shlex
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    subprocess.check_call(["nvidia-smi"])
else:
    print("[提示] 当前运行时未检测到 nvidia-smi（若需要 GPU 前向推理请确保已分配 GPU）。")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("非 Colab 环境，跳过 drive.mount")

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = "https://github.com/wdongdong0708-jpg/BrainDecoding2.git"  # @param {type:"string"}
GIT_BRANCH = "Broderick-EEG"  # @param {type:"string"}
DRIVE_BASE_DIR = "/content/drive/MyDrive/BrainDecoding"  # @param {type:"string"}

DATASET = "broderick2018"
MODALITY = "broderick2018"  # 对应 Google Drive 归档：archives/features_broderick2018.tar

# 评估模式：
#   - "full_bspc_suite" : 一键评估 BSPC 主表全部 4 个模型并导出四模型配对统计检验
#   - "single_config"   : 仅评估下方 CONFIG_FILE + EXP_TAG 指定的单个模型
EVAL_MODE = "full_bspc_suite"  # @param ["full_bspc_suite", "single_config"]
SEED = "0,1,2"  # @param {type:"string"}
SUBSET_ARGS = "--subject all"  # @param {type:"string"}
WINDOW_SECONDS = 1.0
WINDOW_START_OFFSET_SECONDS = None

# 4 个 BSPC 主表模型的 exp_tag（与 colab_gpu_train_broderick.ipynb 默认完全一致）：
WORD_EXP_TAG = "lr4e-4_bs256"  # @param {type:"string"}
CONTEXT_BASELINE_EXP_TAG = "lr4e-4_bs256"  # @param {type:"string"}
CONTEXT_DEBIASED_EXP_TAG = "siamese_debiased_context_preroll"  # @param {type:"string"}
PREFIX_REVISION_EXP_TAG = "causal_null_space_debiased_prefix_m2"  # @param {type:"string"}
PREFIX_REVISION_CONFIG_FILE = "configs/training/siamese_debiased/broderick2018_causal_null_space_prefix_revision.yaml"  # @param {type:"string"}

# 单配置模式参数（当 EVAL_MODE == "single_config" 时生效）：
CONFIG_FILE = "configs/training/siamese_debiased/broderick2018_causal_null_space_prefix_revision.yaml"  # @param {type:"string"}
EXP_TAG = "causal_null_space_debiased_prefix_m2"  # @param {type:"string"}

# 因果负对照与共模扣除控制参数：
CONTROL_CONFIG = "configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml"  # @param {type:"string"}
SHIFT_SECONDS = "-10,-20,-30,-40"  # @param {type:"string"}
SHIFT_MODE = "preroll"  # @param ["preroll", "deterministic_circular"]
CMS_MODE = "causal_ema"  # @param ["causal_ema", "pos_causal_ema", "leave_one_out"]
CONTROL_METHOD = "raw,causal_cms:eta=0.5"  # @param {type:"string"}
CONTROL_METRIC = "vocab50_macro_top10,vocab50_macro_top1,full_vocab_macro_top10,full_vocab_macro_top1"  # @param {type:"string"}

# 四模型配对统计检验各槽位采用的对照方法（若 prefix_revision 使用 amortized_timing，下面会自动切换为 'raw'）：
CONTEXT_DEBIASED_CONTROL_METHOD = "causal_cms:eta=0.5"  # @param {type:"string"}
PREFIX_REVISION_CONTROL_METHOD = "causal_cms:eta=0.5"  # @param {type:"string"}

EVAL_SUBDIR = ""  # @param {type:"string"}
FORCE_REEVAL = False  # @param {type:"boolean"}
RUN_ZERO_SIGNAL_AUDIT = False  # @param {type:"boolean"}
RUN_MEVAL_TRADEOFF_FROM_PT = True  # @param {type:"boolean"}
EXPORT_CROSS_MODEL_COMPARISON = True  # @param {type:"boolean"}
AUTO_UNASSIGN_RUNTIME = True  # @param {type:"boolean"}
# ========================================================

_VALID_EVAL_MODES = ("full_bspc_suite", "single_config")
if EVAL_MODE not in _VALID_EVAL_MODES:
    raise ValueError(f"不支持的 EVAL_MODE={EVAL_MODE!r}，可选：{_VALID_EVAL_MODES}")

if isinstance(SEED, (list, tuple, set)):
    SEED_LIST = [int(s) for s in SEED]
else:
    SEED_LIST = [
        int(s.strip())
        for s in str(SEED).strip("[]()").replace(";", ",").replace(" ", ",").split(",")
        if s.strip()
    ]
SEED_ARG = ",".join(str(s) for s in SEED_LIST)

WINDOW_CLI_ARGS = []
if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != "":
    WINDOW_CLI_ARGS.extend(["--window-seconds", str(WINDOW_SECONDS)])
if WINDOW_START_OFFSET_SECONDS is not None and str(WINDOW_START_OFFSET_SECONDS).strip() != "":
    WINDOW_CLI_ARGS.extend(["--window-start-offset-seconds", str(WINDOW_START_OFFSET_SECONDS)])
SUBSET_CLI_ARGS = shlex.split(str(SUBSET_ARGS or "").strip())


def _resolve_subset_glob(_subset_args: str) -> str:
    _txt = " ".join(str(_subset_args or "").strip().split())
    if not _txt or _txt in ("--subject all", "--subjects all"):
        return "all-*"
    _toks = _txt.split()
    if "--num-subjects" in _toks:
        _idx = _toks.index("--num-subjects")
        if _idx + 1 < len(_toks) and _toks[_idx + 1].isdigit():
            _n = int(_toks[_idx + 1])
            return "sub-01" if _n == 1 else f"subset-n{_n:02d}-*"
    return "*"


SUBSET_GLOB = _resolve_subset_glob(SUBSET_ARGS)


def _infer_stage_from_config(_cfg_path: str) -> str:
    _low = str(_cfg_path).lower()
    if "word_encoder" in _low or "/main_word" in _low:
        return "word"
    if "prefix_revision" in _low:
        return "prefix_revision"
    if "context" in _low:
        return "context"
    return "revision"


def _build_control_cli_args(
    *,
    control_config: str = CONTROL_CONFIG,
    shift_seconds: str | None = SHIFT_SECONDS,
    shift_mode: str | None = SHIFT_MODE,
    cms_mode: str | None = CMS_MODE,
    eval_subdir: str = EVAL_SUBDIR,
) -> list[str]:
    _args = []
    _ctrl = str(control_config or "").strip()
    if _ctrl:
        _args.extend(["--control-config", _ctrl])
        if "zero_signal" not in _ctrl.lower() and shift_seconds is not None and str(shift_seconds).strip() != "":
            _args.extend(["--shift-seconds", str(shift_seconds).strip()])
    if shift_mode is not None and str(shift_mode).strip() != "":
        _args.extend(["--shift-mode", str(shift_mode).strip()])
    if cms_mode is not None and str(cms_mode).strip() != "":
        _args.extend(["--cms-mode", str(cms_mode).strip()])
    if eval_subdir is not None and str(eval_subdir).strip() != "":
        _args.extend(["--eval-subdir", str(eval_subdir).strip()])
    if CONTROL_METHOD is not None and str(CONTROL_METHOD).strip() != "":
        _cm_str = (
            ",".join(str(x).strip() for x in CONTROL_METHOD)
            if isinstance(CONTROL_METHOD, (list, tuple))
            else str(CONTROL_METHOD).strip()
        )
        _args.extend(["--control-method", _cm_str])
    if CONTROL_METRIC is not None and str(CONTROL_METRIC).strip() != "":
        _cmt_str = (
            ",".join(str(x).strip() for x in CONTROL_METRIC)
            if isinstance(CONTROL_METRIC, (list, tuple))
            else str(CONTROL_METRIC).strip()
        )
        _args.extend(["--control-metric", _cmt_str])
    return _args


CONTROL_CLI_ARGS = _build_control_cli_args()

_resolved_prefix_ctrl_method = PREFIX_REVISION_CONTROL_METHOD
if "amortized_timing" in str(PREFIX_REVISION_CONFIG_FILE).lower() or "amortized_timing" in str(PREFIX_REVISION_EXP_TAG).lower():
    if _resolved_prefix_ctrl_method == "causal_cms:eta=0.5":
        _resolved_prefix_ctrl_method = "raw"

if EVAL_MODE == "full_bspc_suite":
    EVAL_JOBS = [
        {
            "slot": "word_only",
            "label": "Word-Only (Stage 1 Baseline)",
            "dataset": DATASET,
            "stage": "word",
            "config_file": "configs/training/broderick2018_word_encoder.yaml",
            "exp_tag": WORD_EXP_TAG,
            "comparison_control_method": "raw",
        },
        {
            "slot": "context_unfrozen",
            "label": "Full-Context Non-Debiased Baseline (Stage 2.1)",
            "dataset": DATASET,
            "stage": "context",
            "config_file": "configs/training/broderick2018_context_transformer.yaml",
            "exp_tag": CONTEXT_BASELINE_EXP_TAG,
            "comparison_control_method": "raw",
        },
        {
            "slot": "context_debiased",
            "label": f"Full-Context Debiased ({CONTEXT_DEBIASED_CONTROL_METHOD})",
            "dataset": DATASET,
            "stage": "context",
            "config_file": "configs/training/siamese_debiased/broderick2018_siamese_debiased_context.yaml",
            "exp_tag": CONTEXT_DEBIASED_EXP_TAG,
            "comparison_control_method": CONTEXT_DEBIASED_CONTROL_METHOD,
        },
        {
            "slot": "prefix_revision",
            "label": f"Prefix-Revision Causal Debiased ({_resolved_prefix_ctrl_method})",
            "dataset": DATASET,
            "stage": "prefix_revision",
            "config_file": PREFIX_REVISION_CONFIG_FILE,
            "exp_tag": PREFIX_REVISION_EXP_TAG,
            "comparison_control_method": _resolved_prefix_ctrl_method,
        },
    ]
    _active_primary_tag = PREFIX_REVISION_EXP_TAG
else:
    _single_stage = _infer_stage_from_config(CONFIG_FILE)
    _single_tag = str(EXP_TAG or "").strip()
    if _single_tag.lower() == "auto":
        _single_tag = Path(CONFIG_FILE).stem
    EVAL_JOBS = [
        {
            "slot": "prefix_revision" if _single_stage == "prefix_revision" else _single_stage,
            "label": f"Single Config ({Path(CONFIG_FILE).stem})",
            "dataset": DATASET,
            "stage": _single_stage,
            "config_file": CONFIG_FILE,
            "exp_tag": _single_tag,
            "comparison_control_method": _resolved_prefix_ctrl_method if _single_stage == "prefix_revision" else "raw",
        }
    ]
    _active_primary_tag = _single_tag

CODE_DIR = "/content/code"
VENV_DIR = "/content/proj-venv"
PYTHON = f"{VENV_DIR}/bin/python"
LINKS_DIR = "/content/studies_links"
LOCAL_CACHE_DIR = "/content/local_cache"

DRIVE_DATA_DIR = f"{DRIVE_BASE_DIR}/data"
DRIVE_ARCHIVE_DIR = f"{DRIVE_BASE_DIR}/archives"
DRIVE_RESULTS_DIR = f"{DRIVE_BASE_DIR}/runs"
DRIVE_MODELS_DIR = f"{DRIVE_BASE_DIR}/models"

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ["BRAINDECODING_DATA_DIR"] = LINKS_DIR
os.environ["BRAINDATA_ROOT"] = LINKS_DIR
os.environ["BRAINDECODING_CACHE_DIR"] = LOCAL_CACHE_DIR
os.environ["BRAINDECODING_ARCHIVE_DIR"] = DRIVE_ARCHIVE_DIR
os.environ["BRAINDECODING_RESULTS_DIR"] = DRIVE_RESULTS_DIR
os.environ["BRAINDECODING_EXP_TAG"] = _active_primary_tag
os.environ["BRAINDECODING_EVAL_SUBDIR"] = str(EVAL_SUBDIR or "").strip()
os.environ["BRAINDECODING_WORD_EXP_TAG"] = WORD_EXP_TAG
os.environ["BRAINDECODING_CONTEXT_EXP_TAG"] = CONTEXT_DEBIASED_EXP_TAG
os.environ["BRAINDECODING_MODEL_DIR"] = DRIVE_MODELS_DIR
os.environ["HF_HOME"] = f"{LOCAL_CACHE_DIR}/huggingface"
os.environ["PYTHONUNBUFFERED"] = "1"

print("DATASET                   =", DATASET)
print("GIT_BRANCH                =", GIT_BRANCH)
print("EVAL_MODE                 =", EVAL_MODE)
print("SEEDS                     =", SEED_LIST, f"(--seed {SEED_ARG})")
print("MODALITY / ARCHIVE        =", f"{MODALITY} -> features_{MODALITY}.tar")
print("CONTROL_CONFIG            =", CONTROL_CONFIG or "(disabled - clean only)")
print("SHIFT_SECONDS             =", SHIFT_SECONDS if str(CONTROL_CONFIG).strip() else "(disabled)")
print("CONTROL_METHOD            =", CONTROL_METHOD)
print("EVAL_SUBDIR (isolated)    =", os.environ["BRAINDECODING_EVAL_SUBDIR"] or "(in-place checkpoint dir)")
print("STAGE 3 LLM BEAM SEARCH   = disabled (per Broderick 2018 word-level BSPC protocol)")
print("QUEUED EVAL_JOBS          =", len(EVAL_JOBS))
for _i, _j in enumerate(EVAL_JOBS, 1):
    print(f"  [{_i}/{len(EVAL_JOBS)}] {_j['label']} -> runs/{_j['dataset']}/{_j['exp_tag']}/{_j['stage']}/... (cfg={_j['config_file']})")

In [ ]:
# 2. 克隆或同步最新代码到 /content/code，初始化虚拟环境并建立 /content/studies_links 软链接桥
import subprocess
import sys
from pathlib import Path

code_path = Path(CODE_DIR)
if not (code_path / ".git").is_dir():
    subprocess.check_call(["git", "clone", "-b", GIT_BRANCH, GIT_REPO_URL, CODE_DIR])
else:
    subprocess.check_call(["git", "-C", CODE_DIR, "fetch", "origin"])
    subprocess.check_call(["git", "-C", CODE_DIR, "checkout", GIT_BRANCH])
    subprocess.check_call(["git", "-C", CODE_DIR, "pull", "--ff-only"])

subprocess.check_call(
    [
        sys.executable,
        "-u",
        str(code_path / "scripts/colab_setup.py"),
        "--venv-dir",
        VENV_DIR,
        "--project-root",
        CODE_DIR,
        "--drive-data-dir",
        DRIVE_DATA_DIR,
        "--links-dir",
        LINKS_DIR,
        "--local-cache-dir",
        LOCAL_CACHE_DIR,
    ]
)

In [ ]:
# 3. 智能解包 archives/features_broderick2018.tar 到 /content/local_cache（已有完整最新 test 报告时自动 0 秒跳过）
import os
import subprocess
from pathlib import Path


def _norm_tag(_tag: str) -> str:
    _t = str(_tag or "").strip()
    return "" if _t.lower() in ("", "default", "root", "none") else _t


def _find_seed_dir(_ds: str, _tag: str, _stage: str, _s: int) -> Path | None:
    _t = _norm_tag(_tag)
    _stage_dir = (Path(DRIVE_RESULTS_DIR) / _ds / _t / _stage) if _t else (Path(DRIVE_RESULTS_DIR) / _ds / _stage)
    if not _stage_dir.exists():
        return None
    for _c in sorted(_stage_dir.glob(f"{SUBSET_GLOB}/seed-{int(_s):03d}")):
        if (_c / "best.ckpt").is_file() or (_c / "best.pt").is_file():
            return _c
    return None


def _job_eval_output_dir(_seed_dir: Path, _stage: str) -> Path:
    _sub = str(EVAL_SUBDIR or "").strip()
    if not _sub:
        return _seed_dir
    # checkpoint_dir = .../dataset/[exp_tag]/stage/subset/seed-XXX -> eval_subdir placed cleanly next to stage
    _stage_parent = _seed_dir.parent.parent.parent
    return _stage_parent / _sub / _stage / _seed_dir.parent.name / _seed_dir.name


def _is_job_test_eval_fresh(_job: dict) -> bool:
    _expect_cms = bool(str(CONTROL_CONFIG or "").strip())
    _subset_dirs = set()
    for _s in SEED_LIST:
        _sd = _find_seed_dir(_job["dataset"], _job["exp_tag"], _job["stage"], _s)
        if _sd is None:
            return False
        _out_sd = _job_eval_output_dir(_sd, _job["stage"])
        _subset_dirs.add(_out_sd.parent)
        _ckpt = _sd / "best.ckpt" if (_sd / "best.ckpt").is_file() else (_sd / "best.pt")
        _eval_json = _out_sd / "evaluation_test.json"
        if not _eval_json.is_file() or _eval_json.stat().st_mtime < _ckpt.stat().st_mtime:
            return False
        if _expect_cms and "zero_signal" not in str(CONTROL_CONFIG).lower():
            _cms_json = _out_sd / "exploratory" / "common_mode_subtraction_test.json"
            if not _cms_json.is_file() or _cms_json.stat().st_mtime < _ckpt.stat().st_mtime:
                return False
    if len(SEED_LIST) > 1:
        for _sub_dir in _subset_dirs:
            if not (_sub_dir / "multi_seed_evaluation_test.json").is_file():
                return False
    return True


ALL_TEST_EVALS_FRESH = (
    (not FORCE_REEVAL)
    and (not RUN_ZERO_SIGNAL_AUDIT)
    and all(_is_job_test_eval_fresh(_j) for _j in EVAL_JOBS)
)

if ALL_TEST_EVALS_FRESH:
    print(f"[0秒跳过解包] 全部 {len(EVAL_JOBS)} 个模型在 seeds={SEED_LIST} 的 split=test 评估结果均已是最新，跳过解包与前向推理！")
else:
    archive_file = Path(DRIVE_ARCHIVE_DIR) / f"features_{MODALITY}.tar"
    if not archive_file.is_file():
        raise FileNotFoundError(
            f"Google Drive 中未找到 Broderick 2018 缓存归档：{archive_file}（期望 features_broderick2018.tar）。请先运行 colab_cpu_cache.ipynb。"
        )
    print(f"[解包归档] {archive_file} ({archive_file.stat().st_size / (1024**3):.2f} GiB) -> {LOCAL_CACHE_DIR}")
    subprocess.check_call(
        [
            PYTHON,
            "-u",
            "scripts/cache_archive.py",
            "unpack",
            "--archive-dir",
            DRIVE_ARCHIVE_DIR,
            "--cache-dir",
            LOCAL_CACHE_DIR,
            "--modality",
            MODALITY,
            "--skip-existing",
        ],
        cwd=CODE_DIR,
        env=os.environ.copy(),
    )
    ds_cache_dir = Path(LOCAL_CACHE_DIR) / "broderick2018"
    required_artifacts = [
        ds_cache_dir / "events/events.csv",
        ds_cache_dir / "signals/manifest.json",
        ds_cache_dir / "text/t5_large_layer_0_5/embeddings.npz",
        ds_cache_dir / "events/manifests/vocabulary_N50.json",
    ]
    for art in required_artifacts:
        if not art.is_file():
            raise FileNotFoundError(f"解包后缺少 Broderick 2018 核心缓存文件：{art}")
        print(f"[OK] {art.relative_to(Path(LOCAL_CACHE_DIR))} ({art.stat().st_size:,} bytes)")

In [ ]:
# 4. 留出测试集 (split=test) 多种子词级评估与因果负对照审计（可选追加 Zero-Signal 全零对照）
import os
import subprocess
from pathlib import Path

EVALUATED_JOBS = []
MISSING_JOBS = []

for _job_idx, _job in enumerate(EVAL_JOBS, 1):
    _job_ds = _job["dataset"]
    _job_tag = _job["exp_tag"]
    _job_stage = _job["stage"]
    _job_cfg = _job["config_file"]
    _missing_seeds = [_s for _s in SEED_LIST if _find_seed_dir(_job_ds, _job_tag, _job_stage, _s) is None]
    print("\n" + "=" * 86)
    print(f"[TEST 评估 {_job_idx}/{len(EVAL_JOBS)}] {_job['label']} | exp_tag={_job_tag} | seeds={SEED_LIST}")
    print("=" * 86)
    if _missing_seeds:
        print(f"[跳过未训练模型] 缺少 seeds={_missing_seeds} 的 best.ckpt/best.pt -> runs/{_job_ds}/{_job_tag}/{_job_stage}/")
        MISSING_JOBS.append((_job, _missing_seeds))
        continue

    EVALUATED_JOBS.append(_job)
    if (not FORCE_REEVAL) and _is_job_test_eval_fresh(_job):
        print(f"[0秒跳过] {_job['label']} 的 split=test 评估与控制指标已是最新。")
        continue

    _cmd = [
        PYTHON,
        "-u",
        "-m",
        "braindecoding",
        "eval-run",
        "--config",
        _job_cfg,
        "--seed",
        SEED_ARG,
        "--split",
        "test",
        "--require-cache",
    ]
    _clean_tag = _norm_tag(_job_tag)
    if _clean_tag:
        _cmd.extend(["--exp-tag", _clean_tag])
    if WORD_EXP_TAG and _job_stage != "word":
        _cmd.extend(["--word-exp-tag", WORD_EXP_TAG])
    _cmd.extend(WINDOW_CLI_ARGS)
    _cmd.extend(CONTROL_CLI_ARGS)
    _cmd.extend(SUBSET_CLI_ARGS)

    _env = os.environ.copy()
    _env["BRAINDECODING_EXP_TAG"] = _clean_tag
    print(f"[执行命令] {' '.join(_cmd)}")
    subprocess.check_call(_cmd, cwd=CODE_DIR, env=_env)

if not EVALUATED_JOBS:
    raise FileNotFoundError(
        f"未找到任何已训练完成的 Broderick 2018 检查点（检查目标：runs/{DATASET}/... seeds={SEED_LIST}）。请先运行 colab_gpu_train_broderick.ipynb。"
    )

if RUN_ZERO_SIGNAL_AUDIT:
    for _job in EVALUATED_JOBS:
        if _job["stage"] == "word" and len(EVALUATED_JOBS) > 1:
            continue
        _zs_cmd = [
            PYTHON,
            "-u",
            "-m",
            "braindecoding",
            "eval-run",
            "--config",
            _job["config_file"],
            "--seed",
            SEED_ARG,
            "--split",
            "test",
            "--require-cache",
            "--control-config",
            "configs/controls/zero_signal/zero_signal_control.yaml",
            "--eval-subdir",
            "zero_signal_audit",
        ]
        _clean_tag = _norm_tag(_job["exp_tag"])
        if _clean_tag:
            _zs_cmd.extend(["--exp-tag", _clean_tag])
        if WORD_EXP_TAG and _job["stage"] != "word":
            _zs_cmd.extend(["--word-exp-tag", WORD_EXP_TAG])
        _zs_cmd.extend(WINDOW_CLI_ARGS)
        _zs_cmd.extend(SUBSET_CLI_ARGS)
        _env_zs = os.environ.copy()
        _env_zs["BRAINDECODING_EXP_TAG"] = _clean_tag
        _env_zs["BRAINDECODING_EVAL_SUBDIR"] = "zero_signal_audit"
        print(f"\n[全零神经信号负对照审计] {_job['label']} -> zero_signal_audit/")
        subprocess.check_call(_zs_cmd, cwd=CODE_DIR, env=_env_zs)

In [ ]:
# 5. 【3 秒 .pt 缓存复评】基于已存 subspace_encoded_cache_test.pt 输出 Pos-Causal EMA 与 Pre-Roll M_eval 权衡曲线
import os
import subprocess
from pathlib import Path
import torch

if RUN_MEVAL_TRADEOFF_FROM_PT:
    _pt_targets = [
        _j for _j in EVALUATED_JOBS if _j["stage"] in ("prefix_revision", "context")
    ] or list(EVALUATED_JOBS)
    _found_any_pt = False
    for _job in _pt_targets:
        _tag_part = _norm_tag(_job["exp_tag"])
        _exp_root = (Path(DRIVE_RESULTS_DIR) / _job["dataset"] / _tag_part) if _tag_part else (Path(DRIVE_RESULTS_DIR) / _job["dataset"])
        _sub_part = str(EVAL_SUBDIR or "").strip()
        _stage_dir = (_exp_root / _sub_part / _job["stage"]) if _sub_part else (_exp_root / _job["stage"])
        if not _stage_dir.exists():
            continue
        _pt_hits = sorted(_stage_dir.glob("*/seed-*/exploratory/subspace_encoded_cache_test.pt"))
        if not _pt_hits:
            continue
        _found_any_pt = True
        _src_pt_dir = _pt_hits[0].parent.parent.parent
        _sample_bundle = torch.load(_pt_hits[0], map_location="cpu", weights_only=False)
        _shift_pool = _sample_bundle.get("shifted_encoded_by_shift") or {}
        _pool_size = len(_shift_pool)
        if _pool_size < 2:
            print(f"[跳过 M_eval 权衡] {_job['label']} 仅包含 {_pool_size} 条错位流（至少需要 2 条）。")
            continue
        _m_grid = [str(m) for m in (1, 2, 4, 6) if m <= _pool_size]
        if not _m_grid:
            print(f"[跳过 M_eval 权衡] {_job['label']} 无匹配的 M_eval 网格（pool_size={_pool_size}）。")
            continue
        _out_tradeoff_dir = (_exp_root / _sub_part / "m_eval_tradeoff" / _job["stage"] / _src_pt_dir.name) if _sub_part else (_exp_root / "m_eval_tradeoff" / _job["stage"] / _src_pt_dir.name)
        print(f"\n[3秒 .pt 缓存复评] {_job['label']} | 错位流总数 pool_size={_pool_size} -> m_eval_grid={','.join(_m_grid)}")
        print(f"  源特征目录 (只读): {_src_pt_dir}")
        print(f"  隔离输出目录     : {_out_tradeoff_dir}")
        subprocess.check_call(
            [
                PYTHON,
                "-u",
                "-m",
                "braindecoding.evaluation.common_mode_subtraction_evaluation",
                "--source-dir",
                str(_src_pt_dir),
                "--output-dir",
                str(_out_tradeoff_dir),
                "--split",
                "test",
                "--pos-etas",
                "0.5,0.8,1.0",
                "--global-etas",
                "0.5",
                "--preroll-etas",
                "0.8,1.0",
                "--m-eval-grid",
                ",".join(_m_grid),
            ],
            cwd=CODE_DIR,
            env=os.environ.copy(),
        )
    if not _found_any_pt:
        print("[提示] 未找到 subspace_encoded_cache_test.pt（若设置了 --shift-seconds none 属正常行为）。")
else:
    print("[跳过 M_eval 权衡] RUN_MEVAL_TRADEOFF_FROM_PT=False。")

In [ ]:
# 6. 【0 秒报表刷新 + 四模型跨模型配对统计检验导出】汇总多种子 Mean ± SD 并导出 multi_seed_cross_model_comparison_test.{md,txt,json}
import sys
from pathlib import Path

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from braindecoding.evaluation.evaluation_reporting import (
    render_saved_evaluation_report,
    resolve_and_export_cross_model_comparison,
    upgrade_and_refresh_saved_multi_seed_reports,
)

UPGRADE_VERBOSE = False
_dataset_root = Path(DRIVE_RESULTS_DIR) / DATASET
_eval_sub_part = str(EVAL_SUBDIR or "").strip()

_targets_to_upgrade = []
for _job in EVALUATED_JOBS:
    _t = _norm_tag(_job["exp_tag"])
    _st = _job["stage"]
    _base_t = (_dataset_root / _t) if _t else _dataset_root
    _p = (_base_t / _eval_sub_part / _st) if _eval_sub_part else (_base_t / _st)
    if _p.exists() and _p not in _targets_to_upgrade:
        _targets_to_upgrade.append(_p)

_all_upgraded_summaries = []
for _t_path in _targets_to_upgrade:
    print(f"[0秒刷新多种子报表] {_t_path}")
    _upgraded_list = upgrade_and_refresh_saved_multi_seed_reports(
        _t_path,
        split="test",
        control_method=CONTROL_METHOD,
        control_metric=CONTROL_METRIC,
        verbose=UPGRADE_VERBOSE,
        print_to_stdout=True,
    )
    _all_upgraded_summaries.extend(_upgraded_list)
    print(f"[完成] 刷新 {len(_upgraded_list)} 份汇总报表（.json / .txt / .md / tables/*.csv）。")

if EXPORT_CROSS_MODEL_COMPARISON:
    _slot_by_name = {_j["slot"]: _j for _j in EVALUATED_JOBS}
    CROSS_MODEL_SPECS = {
        "word_only": {
            "exp_tag": _norm_tag((_slot_by_name.get("word_only") or {}).get("exp_tag", WORD_EXP_TAG)),
            "stage": "word",
            "control_method": (_slot_by_name.get("word_only") or {}).get("comparison_control_method", "raw"),
            "label": "Word-Only (Stage 1 Baseline)",
        },
        "context_unfrozen": {
            "exp_tag": _norm_tag((_slot_by_name.get("context_unfrozen") or {}).get("exp_tag", CONTEXT_BASELINE_EXP_TAG)),
            "stage": "context",
            "control_method": (_slot_by_name.get("context_unfrozen") or {}).get("comparison_control_method", "raw"),
            "label": "Full-Context Non-Debiased Baseline (Stage 2.1)",
        },
        "context_debiased": {
            "exp_tag": _norm_tag((_slot_by_name.get("context_debiased") or {}).get("exp_tag", CONTEXT_DEBIASED_EXP_TAG)),
            "stage": "context",
            "control_method": (_slot_by_name.get("context_debiased") or {}).get("comparison_control_method", CONTEXT_DEBIASED_CONTROL_METHOD),
            "label": f"Full-Context Debiased ({CONTEXT_DEBIASED_CONTROL_METHOD})",
        },
        "prefix_revision": {
            "exp_tag": _norm_tag((_slot_by_name.get("prefix_revision") or {}).get("exp_tag", PREFIX_REVISION_EXP_TAG)),
            "stage": "prefix_revision",
            "control_method": (_slot_by_name.get("prefix_revision") or {}).get("comparison_control_method", _resolved_prefix_ctrl_method),
            "label": f"Prefix-Revision Causal Debiased ({_resolved_prefix_ctrl_method})",
        },
    }
    _primary_tag = _norm_tag(_active_primary_tag)
    if _eval_sub_part:
        _iso_out = (_dataset_root / _primary_tag / _eval_sub_part) if _primary_tag else (_dataset_root / _eval_sub_part)
        _iso_out.mkdir(parents=True, exist_ok=True)
        _cross_out_dirs = [_iso_out]
    else:
        _cross_out_dirs = [_dataset_root]
        if _primary_tag and (_dataset_root / _primary_tag).exists():
            _cross_out_dirs.insert(0, _dataset_root / _primary_tag)

    _cross_res = resolve_and_export_cross_model_comparison(
        _dataset_root,
        cross_model_specs=CROSS_MODEL_SPECS,
        upgraded_summaries=_all_upgraded_summaries,
        output_dirs=_cross_out_dirs,
        split="test",
        verbose=UPGRADE_VERBOSE,
        print_to_stdout=True,
    )
    if _cross_res is not None:
        print(
            f"[跨模型配对检验已导出] multi_seed_cross_model_comparison_test.md -> "
            f"{_cross_res.get('exports', {}).get('cross_model_comparison_md', '')}"
        )

if len(SEED_LIST) == 1:
    for _t_path in _targets_to_upgrade:
        for _ev_json in sorted(_t_path.rglob(f"seed-{int(SEED_LIST[0]):03d}/evaluation_test.json")):
            print(render_saved_evaluation_report(_ev_json, verbose=UPGRADE_VERBOSE))

In [ ]:
# 7. 同步落盘并按 AUTO_UNASSIGN_RUNTIME 决定是否自动释放 GPU 运行时
import os
import time

if hasattr(os, "sync"):
    os.sync()

if AUTO_UNASSIGN_RUNTIME:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        print("评估产物已全部同步至 Google Drive，10 秒后释放运行时以停止扣费...")
        time.sleep(10)
        runtime.unassign()
    except ImportError:
        print("非 Colab 环境，跳过 drive.flush_and_unmount / runtime.unassign。")
else:
    print("AUTO_UNASSIGN_RUNTIME=False：已保留当前 Colab GPU 会话供交互查看。")